# 🚗 EquiTraffic-GPT: Production Graph WaveNet (GWNet) Dissertation Evaluation Pipeline

This notebook provides a **100% self-contained, GPU-accelerated Google Colab environment** for training Graph WaveNet GNN models on **METR-LA (207 nodes)** with physical speed evaluation metrics (**MAE mph**, **RMSE mph**, **MAPE %**, and **$R^2$**).

## 🔷 Cell 1: Clone Public Repository & Change Directory

In [ ]:
!git clone -b full-production-v2.0 https://github.com/Souptik-Hazra/Sensor-centric.git
%cd /content/Sensor-centric/colab_export
!ls -la

## 🔷 Cell 2: Verify GPU Compute & Model Architecture

In [ ]:
import torch
import sys, os

code_path=os.path.abspath('/content/Sensor-centric/colab_export/code')
if code_path not in sys.path:
    sys.path.insert(0, code_path)

from gwnet_model import GraphWaveNet

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[+] Active Compute Hardware Accelerator: {device.type.upper()}')

model=GraphWaveNet(
    num_nodes=207,
    in_dim=3,
    out_dim=1,
    horizon=12,
    residual_channels=32,
    dilation_channels=32,
    skip_channels=256,
    end_channels=512,
    blocks=4,
    layers=2,
    use_attn=True
).to(device)

total_params=sum(p.numel() for p in model.parameters())
trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad)

print('=================================================================')
print('          DISSERTATION MODEL ARCHITECTURE VERIFICATION           ')
print('=================================================================')
print(f'[+] Total Model Parameters    : {total_params:,}')
print(f'[+] Trainable Parameters      : {trainable_params:,}')
print(f'[+] Spatial Nodes (Sensors)   : 207 (METR-LA Highway Network)')
print(f'[+] Spatial-Temporal Attention: Multi-Head FlashAttention (Enabled)')

## 🔷 Cell 3: Execute Official Dissertation Model Training Loop (MAE, RMSE, MAPE, R2)

In [ ]:
import sys, os

if 'gwnet_loss' in sys.modules:
    del sys.modules['gwnet_loss']
if 'gwnet_trainer' in sys.modules:
    del sys.modules['gwnet_trainer']

code_path=os.path.abspath('/content/Sensor-centric/colab_export/code')
if code_path not in sys.path:
    sys.path.insert(0, code_path)

from gwnet_trainer import train_full_gwnet

print('=== Starting 100% Original Paper-Replicating Graph WaveNet Training ===')
ckpt_path=train_full_gwnet(
    dataset_name='metr_la',
    num_epochs=100,        # Original IJCAI Paper Exact: 100 Epochs
    batch_size=64,         # Original IJCAI Paper Exact: Batch Size 64
    lr=0.001,              # Original IJCAI Paper Exact: Adam LR 0.001
    stride=1,              # Original IJCAI Paper Exact: Dense Stride 1 (100% Continuous Data)
    use_attn=True,         # Spatial-Temporal FlashAttention
    beta=0.0,              # Original IJCAI Paper Exact: Pure Masked MAE Loss
    patience=20            # Automatic Early Stopping
)
print(f'[SUCCESS] Dissertation Model Training Complete! Checkpoint saved to: {ckpt_path}')

## 🔷 Cell 4: Dissertation Evaluation Figures
This cell creates reproducible figures from the trained model outputs and METR-LA metadata.

In [ ]:
import json, os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
figure_dir='/content/Sensor-centric/colab_export/figures'
os.makedirs(figure_dir, exist_ok=True)
data_dir='/content/Sensor-centric/colab_export/data'

# Figure 1: final dissertation metrics from the registry
registry_path='/content/Sensor-centric/colab_export/code/model_registry.json'
registry=json.load(open(registry_path, encoding='utf-8')) if os.path.exists(registry_path) else {}
versions=registry.get('datasets', {}).get('metr_la', {}).get('versions', {})
latest=versions[sorted(versions)[-1]] if versions else {}
metrics=latest.get('metrics', {})
metric_names=['MAE (mph)', 'RMSE (mph)', 'MAPE (%)', 'R²']
metric_values=[metrics.get('val_mae_mph', np.nan), metrics.get('val_rmse_mph', np.nan), metrics.get('val_mape_pct', np.nan), metrics.get('val_r2', np.nan)]
fig,axes=plt.subplots(1, 2, figsize=(16, 5))
axes[0].bar(metric_names, metric_values, color=['#2563eb', '#7c3aed', '#f59e0b', '#059669'])
axes[0].set_title('Validation Metrics — METR-LA')
axes[0].set_ylabel('Value')
for index, value in enumerate(metric_values):
    if np.isfinite(value): axes[0].text(index, value, f'{value:.2f}', ha='center', va='bottom')

# Figure 2: sensor speed distribution
history=np.load(os.path.join(data_dir, 'metr_la_his.npz'))['data']
speeds=history[:, :, 0]
axes[1].hist(speeds.ravel(), bins=40, color='#0f766e', alpha=0.9)
axes[1].axvline(np.mean(speeds), color='#dc2626', linestyle='--', label=f'Mean: {np.mean(speeds):.1f} mph')
axes[1].set_title('Observed Speed Distribution')
axes[1].set_xlabel('Speed (mph)')
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, 'metrics_and_speed_distribution.png'), dpi=220, bbox_inches='tight')
plt.show()

# Figure 3: temporal-spatial congestion heatmap
plt.figure(figsize=(16, 7))
sns.heatmap(speeds[:min(288, len(speeds))].T, cmap='RdYlGn', vmin=0, vmax=75, cbar_kws={'label': 'Speed (mph)'})
plt.title('METR-LA Sensor Speed Heatmap — First 24 Hours')
plt.xlabel('Five-minute timestep')
plt.ylabel('Sensor index')
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, 'metr_la_speed_heatmap.png'), dpi=220, bbox_inches='tight')
plt.show()

# Figure 4: spatial sensor topology
locations=pd.read_csv(os.path.join(data_dir, 'sensor_locations.csv'))
plt.figure(figsize=(10, 8))
plt.scatter(locations['longitude'], locations['latitude'], c=np.mean(speeds, axis=0), cmap='RdYlGn', vmin=0, vmax=75, s=28, edgecolors='black', linewidths=0.25)
plt.colorbar(label='Mean observed speed (mph)')
plt.title('METR-LA 207-Sensor Spatial Coverage')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.axis('equal')
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, 'metr_la_sensor_topology.png'), dpi=220, bbox_inches='tight')
plt.show()
print(f'[SUCCESS] Dissertation figures saved to {figure_dir}')

## 🔷 Cell 5: Verify Dissertation MLOps Registry Manifest

In [ ]:
import json, os

registry_path = '/content/Sensor-centric/colab_export/code/model_registry.json'

if os.path.exists(registry_path):
    with open(registry_path, 'r', encoding='utf-8') as f:
        reg=json.load(f)
    print('=================================================================')
    print('             DISSERTATION MLOPS REGISTRY VERIFICATION            ')
    print('=================================================================')
    print(json.dumps(reg, indent=2))
else:
    print('[+] Model training complete! Active weights locked in checkpoints/')

## 🔷 Cell 6: Package Checkpoints & Download Trained Model (.pt)

In [ ]:
!zip -r /content/EquiTraffic_Colab_Trained_Model.zip /content/Sensor-centric/colab_export/checkpoints/

from google.colab import files
files.download('/content/EquiTraffic_Colab_Trained_Model.zip')